
<div style="text-align: center; padding: 30px 10px;">

<h2 style="font-size: 30px; margin-top: 5px;">
Лабораторная работа 1. О смысле жизни.
</h2>

<hr style="width: 60%; border: 1px solid #10069f; margin: 25px auto;">

</div>

Если Вы видите <b><font color="#FF69B4"> Ваш ответ здесь </font></b> - надо поменять текст внутри блока на свой текстовый ответ на вопрос.


ВАЖНО! Здесь и далее от вас ожидается чистый код, близкий к тому, что нужно в индустрии. В контексте данной лабораторной работы ожидаются корректные названия у полей, методов и переменных, которые вы вводите; корректная инкапсуляция - нижнее подчеркивание или два перед названием метода или атрибута. Для корректной инкапсуляции можно (и даже нужно) добавлять нижние подчеркивания даже к названиям методов/атрибутов из условий задач :)

# 1. Абстракции (3 балла)

Напиши абстрактный класс `Organism`. От него будут наследоваться другие классы.

В нём должен быть:

Concrete Methods:

- `property` - `alive`, который проверяет, что текущий возраст не больше, чем максимальный и энергия не отрицательная. При попытке изменения (сеттер), нужно: если статус "живости" не изменяется, то ничего не меняем; если мёртвый -> живой, то возраст - максимальный - 1, энергия - 1; если живой -> мёртвый, то возраст делаем максимальным + 1.
- Мэджик метод, позволяющий использовать экземпляры в if-конструкциях (мэджик метод `__bool__`, связанный с `alive`).
- Инициализатор (`__init__`), который проставляет атрибуты экземпляра: размер популяции `population_size = 0`, текущий возраст `current_age = 0`, `current_energy = 1` и `id` в зависимости от текущей глобальной переменной `ID_COUNTER`. Последнюю надо увеличить на 1, чтобы последующие ID отличались.
- Метод класса (`classmethod`) `spawn`, принимающий на вход `population_size`, `current_age` и фиксирующий их для объекта.
- Метод `reproduce`, который возвращает новый объект данного класса, в котором `current_age = 0`, `current_energy` - меньше в `REPRODUCE_NEW_COEF` раз, округлённых вниз; population_size - меньше в `REPRODUCE_POPULATION_COEF` раз, округлённых вниз. Кроме того, энергия изначального объекта понижается в `REPRODUCE_OLD_COEF` раз и логгирует в поток вывода сообщение с ID тех, кто размножился и тех, кто появился.


Abstract Methods:

- Мэджик метод, отвечающий за репрезентацию объекта (str/repr)
- Метод `replenish_energy`, отвечающий за пополнение энергии



In [1]:
from abc import ABC, abstractmethod

class Organism(ABC):
    ID_COUNTER = 1
    REPRODUCE_NEW_COEF = 2
    REPRODUCE_POPULATION_COEF = 10
    REPRODUCE_OLD_COEF = 3

    @abstractmethod
    def __repr__(self): ...

    @abstractmethod
    def replenish_energy(self): ...

    def __init__(self, *args, **kwargs):
        self._population_size = 0
        self._current_age = 0
        self._current_energy = 1
        self._id = Organism.ID_COUNTER
        Organism.ID_COUNTER += 1

    @property
    def alive(self) -> bool:
        return self._current_age <= self._max_age and self._current_energy >= 0

    @alive.setter
    def alive(self, life):
        if self.alive != life:
            if life == True:
                self._current_age = self._max_age - 1
                self._current_energy = 1
            else:
                self._current_age = self._max_age + 1

    def __bool__(self):
        return self.alive

    @classmethod
    def spawn(cls, population_size, current_age):
        creature = cls()
        creature._population_size = population_size
        creature._current_age = current_age
        return creature


    def reproduce(self):
        new_organism = self.__class__()
        new_organism._population_size = self._population_size // Organism.REPRODUCE_POPULATION_COEF
        new_organism._current_age = 0
        new_organism._current_energy = self._current_energy // Organism.REPRODUCE_NEW_COEF
        self._current_energy //= Organism.REPRODUCE_OLD_COEF
        print(f"Организм {self._id} создал нового организма {new_organism._id}")
        return new_organism


# 2. Concrete implementations (2 балла)

Реализуй два класса - Herbivore (травоядные) и Plant (растения). Оба должны наследоваться от Organism.

Для Herbivore:

- Инициализатор, который проставляет `max_age` равным `MAX_HERBIVORE_AGE`
- `replenish_energy(eaten_plant_energy)`, который увеличивает количество энергии на `eaten_plant_energy`.


Для Plant:

- Инициализатор, который проставляет `max_age` равным `MAX_PLANT_AGE`
- `replenish_energy` увеличивает текущий запас энергии на `PLANT_ENERGY_INC_COEF`

In [2]:
class Herbivore(Organism):
    MAX_HERBIVORE_AGE = 30

    def __repr__(self):
        return f"Herbivore(id={self._id}, age={self._current_age}, energy={self._current_energy}, pop={self._population_size})"

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._max_age = Herbivore.MAX_HERBIVORE_AGE

    def replenish_energy(self, eaten_plant_energy):
        self._current_energy += eaten_plant_energy

class Plant(Organism):
    MAX_PLANT_AGE = 5
    PLANT_ENERGY_INC_COEF = 10

    def __repr__(self):
        return f"Plant(id={self._id}, age={self._current_age}, energy={self._current_energy}, pop={self._population_size})"

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._max_age = Plant.MAX_PLANT_AGE

    def replenish_energy(self):
        self._current_energy += Plant.PLANT_ENERGY_INC_COEF


Напиши несколько примеров работы с этими классами - протестируй таким образом, что всё ок

In [3]:
h = Herbivore()
p = Plant()
print(h, p)

h.replenish_energy(10)
p.replenish_energy()
print("После пополнения:", h, p)

h2 = h.reproduce()
p2 = p.reproduce()
print("После размножения:")
print("Родитель h2:", h)
print("Потомок h:", h2)
print("Родитель p2:", p)
print("Потомок p:", p2)

if h:
    print("h жив")
if not p2.alive:
    print("p2 мёртв")

h._current_age = 150
print("h жив?", h.alive)
h.alive = True
print("После оживления:", h)

Herbivore(id=1, age=0, energy=1, pop=0) Plant(id=2, age=0, energy=1, pop=0)
После пополнения: Herbivore(id=1, age=0, energy=11, pop=0) Plant(id=2, age=0, energy=11, pop=0)
Организм 1 создал нового организма 3
Организм 2 создал нового организма 4
После размножения:
Родитель h2: Herbivore(id=1, age=0, energy=3, pop=0)
Потомок h: Herbivore(id=3, age=0, energy=5, pop=0)
Родитель p2: Plant(id=2, age=0, energy=3, pop=0)
Потомок p: Plant(id=4, age=0, energy=5, pop=0)
h жив
h жив? False
После оживления: Herbivore(id=1, age=29, energy=1, pop=0)


В каком порядке должен вызываться (и вызываются у тебя):
- метод `spawn`
- `__init__` у ребёнка (`Plant/Herbivore`)
- `__init__` у родителя (`Organism`)?

<b><font color="#ff0000"> Ответ:

1) метод `spawn`
2) `__init__` у родителя (`Organism`)
3) `__init__` у ребёнка (`Plant/Herbivore`)

# 3. Нюансы работы с классами (3 балла)


В следующих ячейках изучи вопросы копирования классов как объектов (НЕ экземпляров! Именно самих классов) и последующего их сравнения. Также изучи, можно ли поменять `__bases__` и/или `__mro__` после создания класса как объекта. Если что-то сходу не получается - погугли :)

<b><font color="#ff0000"> Ответ:

In [4]:
a = [1, 2]
b = a

b[1] = 3
a, b


([1, 3], [1, 3])

<b><font color="#ff0000"> В данном случае у нас a и b - это просто одинаковое название одного и того же объекта, поэтому если мы изменяем один объект, то второй также изменяется

In [5]:
a = [1, [2]]
b = a.copy()

a[1].append(3)
a, b


([1, [2, 3]], [1, [2, 3]])

<b><font color="#ff0000"> Здесь у нас скопировался внешний массив, которые имеют разные `id`, но внутренний массив `[2]` одинаковый, поэтому при изменении этого массива в a или b он поменяется и в другом объекте.

Решение есть - можно использовать deepcopy() из библиотеки copy для полного копирования объекта, без одновременного изменения данных.

In [6]:
from copy import deepcopy

a = [1, [2]]
b = deepcopy(a)

a[1].append(3)
a, b

([1, [2, 3]], [1, [2]])

<b><font color="#ff0000"> MRO - порядок инициализации классов, который используется при наследовании и поиска методов и атрибутов. Он строится по конкретному алгоритму (и по нескольким правилам, такие как: родитель должен следовать позже ребёнка; каждый класс встречается ровно 1 раз), поэтому порядок классов не более, чем единственный. Его может не существовать, если, например, наследование зацикливается. В таком случае выходит ошибка исполнения.

`__bases__` - массив/список прямых предков данного класса-объекта.

MRO напрямую изменить нельзя, так как он строится по алгоритму.

`__bases__` изменить можно, но так делать не надо. При изменении `__bases__` может измениться порядок MRO.

Какие выводы ты можешь сделать? Какие есть нюансы:

1. Копирования классов?

2. Сравнения классов?

3. Изменения `__bases__` и `__mro__` после создания класса?

<b><font color="#ff0000"> Ответ:
1. Копирование объектов класса нужно делать с осторожностью, иначе мы можем долго искать, где на самом деле затаилась какая-то ошибка в выводе. Выше разобраны некоторые такие случаи
2. Классы можно сравнивать по-разному:
Оператор `is` проверяет `id` соответсвующих классов, то есть полное их совпадение без копирований;
Команда (мэджик-метод) `__eq__` сравнивает классы по заданным параметрам. Например, равенство значений атрибутов, или равенство `__dict__`, `__bases__ `и тп.
Поэтому нужно заранее понимать, в каком смысле два класса равны/эквивалентны друг к другу.
3. Изменять `__bases__` и `__mro__` вообще лучше не стоит, так как это делать опасно и некрасиво. Второе вообще напрямую нельзя изменить (сказано выше).

Теперь изучи следующие два примера и проанализируй.

In [ ]:
class A:
    pass

class B:
    pass

B.__bases__ = (A, )


In [8]:
class A:
    pass

class B:
    pass

class C(A):
    pass

C.__bases__ = (B, )


Выводы:

<b><font color="#ff0000"> `__bases__` менять можно не всегда. Можно, когда класс не наследуется от встроенных типов данных, и если после изменения существует `__mro__`.

Теперь попробуй посоздавать экземпляры классов и поменять у них классы динамически. Всегда ли это получается? Почему?

In [ ]:
class A:
    def __init__(self, list):
        self.x = list[0]
        self.y = list[1]

    def __add__(self, other):
        self.x += other.x
        self.y += other.y

class B(A):
    def __init__(self, list):
        self.x = list[0]
        self.y = list[1]

    def __mul__(self, other):
        self.x = self.x * other.x - self.y * other.y
        self.y = self.x * other.y + self.y * other.x

a = A([4, 6])
a.__class__ = B
a += A([1, 2])
a.__class__ = int
a *= A([4, 1])

Выводы:

<b><font color="#ff0000"> Присваивание поддерживается только для изменяемых типов или подклассов ModuleType.

# 4. Mixin'ы. (5 баллов)


Наша цель - в Runtime добавлять к классам новые характеристики, логика которых будет описана в Mixin'ах.

Подумай, как надо поменять реализацию классов `Herbivore` и `Plant`, чтобы легко вызывались методы из Mixin'ов, но при этом если их нет, тоже ничего не ломалось.

Как это сделать?

<b><font color="#ff0000"> Да, мы можем выносить некоторые методы и функции в отдельные классы mixin'ы и потом наследоваться от них. Но если каких-то методов, которые мы вызываем, нет, то можно написать функцию `__getattr__`, которая будет работать в неопределённых ситуациях.

Скопируй `Herbivore` и `Plant` и добавь изменения в них, чтобы работало корректно в будущем

In [10]:
class Herbivore(Organism):
    MAX_HERBIVORE_AGE = 30

    def __repr__(self):
        return f"Herbivore(id={self._id}, age={self._current_age}, energy={self._current_energy}, pop={self._population_size})"

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._max_age = Herbivore.MAX_HERBIVORE_AGE

    def replenish_energy(self, eaten_plant_energy):
        self._current_energy += eaten_plant_energy


class Plant(Organism):
    MAX_PLANT_AGE = 5
    PLANT_ENERGY_INC_COEF = 10

    def __repr__(self):
        return f"Plant(id={self._id}, age={self._current_age}, energy={self._current_energy}, pop={self._population_size})"

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._max_age = Plant.MAX_PLANT_AGE

    def replenish_energy(self):
        self._current_energy += Plant.PLANT_ENERGY_INC_COEF

Напиши следующие Mixin'ы:

- `ShortLifespan` - уменьшает максимальный возраст организма в `SHORT_LIFESPAN_COEF` раз, но увеличивает прибавку энергии в `SHORT_LIFESPAN_ENERGY_COEF` раз

- `LongLifespan` - обратное предыдущему. Добавь соответствующие константы.

- `HighFertilityMixin` - увеличивает у возвращаемого "ребёнка" в методе `reproduce` `population_size` в `HIGH_FERTILITY_COEF` раз, но повышает затраты энергии в `reproduce` в `HIGH_FERTILITY_ENERGY_COEF` раз.

- `Toxic` (только для растений) - при поедании такого растения количество животных из данного набора уменьшается в `TOXIC_COEF` раз.

- `IdentifiesToxicPlants` (только для травоядных) - при выборе растения в фазе поедания, если растение токсично, выбирают не есть его.

- `ResilientToToxicPlants` (только для травоядных) - при выборе токсичного растения смертность уменьшена в `RESILIENT_COEF` раз.

In [11]:
class ShortLifespan:
    SHORT_LIFESPAN_COEF = 2
    SHORT_LIFESPAN_ENERGY_COEF = 2

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._max_age //= ShortLifespan.SHORT_LIFESPAN_COEF
        self._current_energy *= self.SHORT_LIFESPAN_ENERGY_COEF

class LongLifespan:
    LONG_LIFESPAN_COEF = 2
    LONG_LIFESPAN_ENERGY_COEF = 2

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._max_age *= LongLifespan.LONG_LIFESPAN_COEF
        self._current_energy //= LongLifespan.LONG_LIFESPAN_ENERGY_COEF

class HighFertilityMixin:
    HIGH_FERTILITY_COEF = 2
    HIGH_FERTILITY_ENERGY_COEF = 2

    def reproduce(self):
        creature = super().reproduce()
        creature._population_size *= HighFertilityMixin.HIGH_FERTILITY_COEF
        self._current_energy //= HighFertilityMixin.HIGH_FERTILITY_ENERGY_COEF
        return creature

class Toxic:
    TOXIC_COEF = 3

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._toxic = True

    def eat_toxic_plant(self, herbivore):
        herbivore._population_size //= Toxic.TOXIC_COEF

class IdentifiesToxicPlants:
    def choose_plant(self, plants):
        safe_plants = [pl for pl in plants if not pl._toxic]
        if safe_plants:
            return super().choose_plant(safe_plants)
        return None

class ResilientToToxicPlants:
    RESILIENT_COEF = 4

    def eat(self, plant):
        self._population_size //= ResilientToToxicPlants.RESILIENT_COEF


Перекопируй код для абстрактного класса из п. 1 данной работы и добавь к нему метод `mutate`: с вероятностью `MUTATION_POSSIBILITY` процентов он выбирает какой-то из mixin'ов, который у него ещё не стоит в родителях, и добавляет его в родителей; иначе - он выбирает одного из родителей и удаляет его из кортежа родителей (`__bases__`).


Подумай, как это реализовать так, чтобы сам метод был общий и для `Plant`, и для `Herbivore`, но при этом выбор возможных Mixin'ов был разный. При надобности их тоже скопируй ниже и поправь в них код.


Также важно! в методе `mutate` должно быть логгирование (`print` с описанием произошедшего), чтобы было понятно, кто мутировал (и во что).

In [12]:
from abc import ABC, abstractmethod
import random

class Organism(ABC):
    ID_COUNTER = 1
    REPRODUCE_NEW_COEF = 2
    REPRODUCE_POPULATION_COEF = 10
    REPRODUCE_OLD_COEF = 3
    MUTATION_POSSIBILITY = 0.322

    @abstractmethod
    def __repr__(self): ...

    @abstractmethod
    def replenish_energy(self): ...

    def __init__(self, *args, **kwargs):
        self._population_size = 0
        self._current_age = 0
        self._current_energy = 1
        self._id = Organism.ID_COUNTER
        Organism.ID_COUNTER += 1

    def get_max_age(self):
        return self._max_age

    @property
    def alive(self) -> bool:
        return self._current_age <= self._max_age and self._current_energy >= 0

    @alive.setter
    def alive(self, life):
        if self.alive != life:
            if life == True:
                self._current_age = self._max_age - 1
                self._current_energy = 1
            else:
                self._current_age = self._max_age + 1

    def __bool__(self):
        return self.alive

    @classmethod
    def spawn(cls, population_size, current_age):
        creature = cls()
        creature._population_size = population_size
        creature._current_age = current_age
        return creature


    def reproduce(self):
        new_organism = self.__class__()
        new_organism._population_size = self._population_size // Organism.REPRODUCE_POPULATION_COEF
        new_organism._current_age = 0
        new_organism._current_energy = self._current_energy // Organism.REPRODUCE_NEW_COEF
        self._current_energy //= Organism.REPRODUCE_OLD_COEF
        print(f"Организм {self._id} создал новый организм {new_organism._id}")
        return new_organism

    def mutate(self):
        old_cls = self.__class__
        mixins = old_cls.available_mixins

        current_bases = old_cls.__bases__
        if random.uniform(0, 1) < Organism.MUTATION_POSSIBILITY:
            candidates = [m for m in mixins if m not in current_bases]
            if not candidates:
                print(f"Организм {self._id} имеет уже все mixin'ы")
                return
            mixin = random.choice(candidates)
            new_bases = (mixin, ) + current_bases
            print(f"К организму {self._id} добавился mixin {mixin.__name__}")
        else:
            candidates = [m for m in current_bases if m in mixins]
            if not candidates:
                print(f"Организм {self._id} не имеет mixin'ов")
                return
            mixin = random.choice(candidates)
            new_bases = tuple(m for m in current_bases if m != mixin)
            print(f"Из организма {self._id} удалился mixin {mixin.__name__}")

        new_name = f"{old_cls.__name__}_mutated_{self._id}"
        attrs = dict(old_cls.__dict__)
        for key in ['__dict__']:
            if key in attrs:
                del attrs[key]
        new_cls = type(new_name, new_bases, attrs)
        # self.__class__ = new_cls




class Herbivore(Organism):
    MAX_HERBIVORE_AGE = 30
    available_mixins = [ShortLifespan, LongLifespan, HighFertilityMixin,
                        IdentifiesToxicPlants, ResilientToToxicPlants]

    def __repr__(self):
        return f"Herbivore(id={self._id}, age={self._current_age}, energy={self._current_energy}, pop={self._population_size})"

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._max_age = Herbivore.MAX_HERBIVORE_AGE

    def replenish_energy(self, eaten_plant_energy):
        self._current_energy += eaten_plant_energy

    def choose_plant(self, plants):
        if plants:
            return random.choice(plants)
        return None


class Plant(Organism):
    MAX_PLANT_AGE = 5
    PLANT_ENERGY_INC_COEF = 10
    available_mixins = [ShortLifespan, LongLifespan, HighFertilityMixin, Toxic]

    def __repr__(self):
        return f"Plant(id={self._id}, age={self._current_age}, energy={self._current_energy}, pop={self._population_size})"

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._max_age = Plant.MAX_PLANT_AGE

    def replenish_energy(self):
        self._current_energy += Plant.PLANT_ENERGY_INC_COEF


Напиши пару примеров работы с `mutate`, чтобы протестировать, что всё сделано хорошо.

In [13]:
if __name__ == "__main__":
    # Создаём несколько организмов без мутаций
    h1 = Herbivore()
    h2 = Herbivore()
    p1 = Plant()
    p2 = Plant()

    print("Начальное состояние:")
    print("h1:", h1, "bases:", [b.__name__ for b in h1.__class__.__bases__])
    print("p1:", p1, "bases:", [b.__name__ for b in p1.__class__.__bases__])

    print("\n--- Мутации ---")
    # Несколько раз мутируем h1
    for i in range(3):
        h1.mutate()
        print(f"После мутации {i+1}: h1 класс = {h1.__class__.__name__}, bases = {[b.__name__ for b in h1.__class__.__bases__]}")

    print("\n--- Мутация растения ---")
    p1.mutate()
    print(f"p1 класс = {p1.__class__.__name__}, bases = {[b.__name__ for b in p1.__class__.__bases__]}")
    if hasattr(p1, 'toxic'):
        print("p1 стал токсичным!")

    print("\n--- Проверка работы миксинов ---")
    # Если у h1 есть IdentifiesToxicPlants, он должен избегать токсичных растений
    # Для простоты проверим наличие метода choose_plant
    if hasattr(h1, 'choose_plant'):
        print("h1 умеет выбирать растения (возможно, с учётом токсичности)")
    else:
        print("h1 не умеет выбирать растения (базовый Herbivore)")

    # Проверим метод reproduce у h2 (может быть HighFertilityMixin)
    h2.mutate()
    print("h2 после мутации:", h2.__class__.__name__, "bases:", [b.__name__ for b in h2.__class__.__bases__])
    offspring = h2.reproduce()
    print("h2 произвёл потомка:", offspring)

Начальное состояние:
h1: Herbivore(id=1, age=0, energy=1, pop=0) bases: ['Organism']
p1: Plant(id=3, age=0, energy=1, pop=0) bases: ['Organism']

--- Мутации ---
Организм 1 не имеет mixin'ов
После мутации 1: h1 класс = Herbivore, bases = ['Organism']
Организм 1 не имеет mixin'ов
После мутации 2: h1 класс = Herbivore, bases = ['Organism']
К организму 1 добавился mixin LongLifespan
После мутации 3: h1 класс = Herbivore, bases = ['Organism']

--- Мутация растения ---
К организму 3 добавился mixin Toxic
p1 класс = Plant, bases = ['Organism']

--- Проверка работы миксинов ---
h1 умеет выбирать растения (возможно, с учётом токсичности)
Организм 2 не имеет mixin'ов
h2 после мутации: Herbivore bases: ['Organism']
Организм 2 создал новый организм 5
h2 произвёл потомка: Herbivore(id=5, age=0, energy=0, pop=0)


# 5. Собираем симуляцию вместе (5 баллов)


Напиши класс `Game`, который будет хранить список текущих животных и растений. Заполни его на своё усмотрение в начале игры (метод `__init__`), НО в начале не должно быть ни одного животного/растения с мутациями (mixin'ами).

У него должны быть:

- метод `__init__` (см. выше)

- метод `correct_age_energy`, который итерируется по всем живым организмам и делает с каждым следующие шаги:

1. Вычитает константу потребления энергии (они различны для животных и растений)
2. Прибавляет age на 1.
3. Если животное или растение стало слишком старым ИЛИ энергия стала <= 0, то оно умирает (и это логируется в поток вывода).


- метод `cycle_through_actions`, который итерируется по всем живым организмам и с каждым равновероятно делает одно из трёх действий: `replenish_energy` (для животных оно выбирает растение и уменьшает его популяцию в `PLANT_EATEN_COEF` и энергию в `PLANT_ENERGY_EATEN_COEF` раз, после чего повышает собственную энергию через `replenish_energy`), `reproduce`, `mutate`. Каждый из этих трёх действий должно корректно логироваться (чтобы мы понимали, что происходит в симуляции).


- метод `play_game(n)` - который запускает игру и делает последовательно `n` циклов жизни (запусков `correct_age_energy` и `cycle_through_actions`).

In [14]:
# TODO

Собери все возможные коэффициенты и установи их на своё усмотрение в ячейке ниже.

Запусти игру. Какие результаты получились после запуска?)

In [15]:
# TODO

Результаты

<b><font color="#FF69B4"> Ваш ответ здесь </font></b>